In [ ]:
!pip install -qU langchain-qdrant langchain-pymupdf4llm langchain langchain-huggingface pytesseract Pillow langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 56.9 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

qdrant_key = userdata.get('QDRANT_API_KEY')
qdrant_cloud_url = userdata.get('QDRANT_URL')
hf_key = userdata.get('HF_TOKEN')

In [ ]:
"""
Parent-Child Chunking RAG Pipeline — Hybrid Dense + Sparse (Server-Side BM25)
==============================================================================

Architecture:
- OCR extracts text per PDF page
- Parent chunks (large, ~2000 chars) split further into child chunks (~512 chars)
- Two Qdrant collections:
    * student_handbook_child  -> dense (HF endpoint embeddings) + sparse (Qdrant server-side BM25)
    * student_handbook_parent -> dummy 1-dim vector, accessed only via client.retrieve() by ID
- Search: hybrid RSF fusion over child collection -> dedup by parent_id -> resolve full context from parent collection

Requires: Qdrant Cloud cluster with "Inference" enabled and BM25 listed under
Dense/Sparse Text Embedding Models (confirmed in your cluster's Inference tab).
No local sparse-embedding library is needed — Qdrant computes BM25 vectors server-side.
"""

import io
import uuid

import fitz  # PyMuPDF
import pytesseract
from PIL import Image
from tqdm import tqdm

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpointEmbeddings

from qdrant_client import QdrantClient, models

# ---------------------------------------------------------------------------
# 0. Configuration — fill these in
# ---------------------------------------------------------------------------
PDF_PATH = "/content/students-handbook-2019-20.pdf"
HF_API_TOKEN = hf_key                 # your existing HuggingFace token variable
QDRANT_URL = qdrant_cloud_url         # your existing Qdrant Cloud URL variable
QDRANT_API_KEY = qdrant_key           # your existing Qdrant API key variable

CHILD_COLLECTION = "student_handbook_child"
PARENT_COLLECTION = "student_handbook_parent"

DENSE_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
DENSE_VECTOR_SIZE = 768
SPARSE_MODEL_NAME = "qdrant/bm25"      # exact string shown in your cluster's Inference tab

PARENT_CHUNK_SIZE = 2000
PARENT_CHUNK_OVERLAP = 200
CHILD_CHUNK_SIZE = 512
CHILD_CHUNK_OVERLAP = 50

BATCH_SIZE = 16


# ---------------------------------------------------------------------------
# 1. OCR extraction
# ---------------------------------------------------------------------------
def ocr_pdf(pdf_path: str, dpi: int = 300) -> list[Document]:
    """
    Render each PDF page as an image and run Tesseract OCR on it.
    Returns a list of LangChain Documents with page_content filled.
    """
    pdf = fitz.open(pdf_path)
    docs = []

    for page_num in range(len(pdf)):
        page = pdf[page_num]
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat)
        img = Image.open(io.BytesIO(pix.tobytes("png")))

        text = pytesseract.image_to_string(img, lang="eng").strip()

        docs.append(Document(
            page_content=text,
            metadata={
                "source": pdf_path,
                "page_number": page_num,
                "total_pages": len(pdf),
            }
        ))

        if page_num % 5 == 0:
            print(f"  OCR progress: {page_num + 1}/{len(pdf)} pages")

    pdf.close()
    print(f"✅ OCR complete — {len(docs)} pages extracted")
    return docs


# ---------------------------------------------------------------------------
# 2. Parent + child splitting
# ---------------------------------------------------------------------------
def build_parent_child_chunks(docs: list[Document]):
    """
    Splits documents into parent chunks, then each parent into child chunks.
    Returns:
        parent_records: list of dicts {id, text, page_number, source} for the parent collection
        child_chunks:   list of LangChain Documents with metadata["parent_id"] set
    """
    parent_splitter = RecursiveCharacterTextSplitter(
        chunk_size=PARENT_CHUNK_SIZE,
        chunk_overlap=PARENT_CHUNK_OVERLAP,
        separators=["\n\n", "\n", ".", " "]
    )
    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHILD_CHUNK_SIZE,
        chunk_overlap=CHILD_CHUNK_OVERLAP,
        separators=["\n\n", "\n", ".", " "]
    )

    parent_docs = parent_splitter.split_documents(docs)
    parent_docs = [p for p in parent_docs if len(p.page_content.strip()) > 30]

    parent_records = []
    child_chunks = []

    for parent in parent_docs:
        parent_id = str(uuid.uuid4())
        parent_records.append({
            "id": parent_id,
            "text": parent.page_content,
            "page_number": parent.metadata.get("page_number", -1),
            "source": parent.metadata.get("source", ""),
        })

        children = child_splitter.split_documents([parent])
        for child in children:
            child.metadata["parent_id"] = parent_id
            child_chunks.append(child)

    child_chunks = [c for c in child_chunks if len(c.page_content.strip()) > 30]
    print(f"✅ {len(parent_records)} parent chunks → {len(child_chunks)} child chunks")
    return parent_records, child_chunks


# ---------------------------------------------------------------------------
# 3. Qdrant setup
# ---------------------------------------------------------------------------
def create_collections(client: QdrantClient):
    client.create_collection(
        collection_name=CHILD_COLLECTION,
        vectors_config=models.VectorParams(size=DENSE_VECTOR_SIZE, distance=models.Distance.COSINE),
        sparse_vectors_config={
            "text-sparse": models.SparseVectorParams(
                index=models.SparseIndexParams(on_disk=False)
            )
        },
    )
    print(f"✅ Child collection '{CHILD_COLLECTION}' created (dense + server-side BM25 sparse)")

    client.create_collection(
        collection_name=PARENT_COLLECTION,
        vectors_config=models.VectorParams(size=1, distance=models.Distance.COSINE),
    )
    print(f"✅ Parent collection '{PARENT_COLLECTION}' created (dummy vector, ID lookup only)")


# ---------------------------------------------------------------------------
# 4. Insertion
# ---------------------------------------------------------------------------
def insert_parents(client: QdrantClient, parent_records: list[dict]):
    parent_points = [
        models.PointStruct(
            id=p["id"],
            vector=[0.0],
            payload={
                "text": p["text"],
                "page_number": p["page_number"],
                "source": p["source"],
            },
        )
        for p in parent_records
    ]
    client.upsert(collection_name=PARENT_COLLECTION, points=parent_points)
    print(f"✅ {len(parent_points)} parent chunks inserted")


def insert_child_chunks_hybrid(client: QdrantClient, embeddings, chunks: list[Document]):
    for start in tqdm(range(0, len(chunks), BATCH_SIZE)):
        batch = chunks[start: start + BATCH_SIZE]
        texts = [c.page_content for c in batch]

        dense_vecs = embeddings.embed_documents(texts)  # HuggingFace endpoint, still client-side

        points = [
            models.PointStruct(
                id=str(uuid.uuid4()),
                vector={
                    "": dense_vecs[i],
                    "text-sparse": models.Document(text=chunk.page_content, model=SPARSE_MODEL_NAME),
                },
                payload={
                    "text": chunk.page_content,
                    "parent_id": chunk.metadata["parent_id"],
                    "page_number": chunk.metadata.get("page_number", -1),
                    "source": chunk.metadata.get("source", ""),
                },
            )
            for i, chunk in enumerate(batch)
        ]
        client.upsert(collection_name=CHILD_COLLECTION, points=points)

    print(f"✅ {len(chunks)} child chunks inserted (dense + server-side BM25 sparse)")


# ---------------------------------------------------------------------------
# 5. Hybrid search
# ---------------------------------------------------------------------------
def hybrid_search(client: QdrantClient, embeddings, query: str, top_k: int = 5):
    dense_query_vector = embeddings.embed_query(query)

    results = client.query_points(
        collection_name=CHILD_COLLECTION,
        prefetch=[
            models.Prefetch(query=dense_query_vector, limit=top_k * 2),
            models.Prefetch(
                query=models.Document(text=query, model=SPARSE_MODEL_NAME),
                using="text-sparse",
                limit=top_k * 2,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=top_k * 2,
        with_payload=True,
    )

    ordered_parent_ids, seen = [], set()
    child_scores, child_snippets = {}, {}

    for r in results.points:
        pid = r.payload["parent_id"]
        if pid not in seen:
            seen.add(pid)
            ordered_parent_ids.append(pid)
            child_scores[pid] = r.score
            child_snippets[pid] = r.payload["text"]
        if len(ordered_parent_ids) == top_k:
            break

    parents = client.retrieve(
        collection_name=PARENT_COLLECTION,
        ids=ordered_parent_ids,
        with_payload=True,
    )
    parent_map = {p.id: p.payload for p in parents}

    return [
        {
            "matched_snippet": child_snippets[pid],
            "parent_context": parent_map[pid]["text"],
            "page": parent_map[pid]["page_number"],
            "score": child_scores[pid],
        }
        for pid in ordered_parent_ids
    ]

In [ ]:
docs = ocr_pdf(PDF_PATH)

  OCR progress: 1/56 pages
  OCR progress: 6/56 pages
  OCR progress: 11/56 pages
  OCR progress: 16/56 pages
  OCR progress: 21/56 pages
  OCR progress: 26/56 pages
  OCR progress: 31/56 pages
  OCR progress: 36/56 pages
  OCR progress: 41/56 pages
  OCR progress: 46/56 pages
  OCR progress: 51/56 pages
  OCR progress: 56/56 pages
✅ OCR complete — 56 pages extracted


In [ ]:
parent_records, child_chunks = build_parent_child_chunks(docs)

✅ 75 parent chunks → 238 child chunks


In [ ]:
# -- Embeddings client (dense only — sparse is server-side) --
embeddings = HuggingFaceEndpointEmbeddings(
        huggingfacehub_api_token=HF_API_TOKEN,
        model=DENSE_MODEL_NAME,
)

In [ ]:
# -- Qdrant client + collections --
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
create_collections(client)

✅ Child collection 'student_handbook_child' created (dense + server-side BM25 sparse)
✅ Parent collection 'student_handbook_parent' created (dummy vector, ID lookup only)


In [ ]:
# -- Insert --
insert_parents(client, parent_records)
insert_child_chunks_hybrid(client, embeddings, child_chunks)

✅ 75 parent chunks inserted


100%|██████████| 15/15 [00:18<00:00,  1.23s/it]

✅ 238 child chunks inserted (dense + server-side BM25 sparse)


In [ ]:
# -- Test search --
print("\n--- Test query ---")
results = hybrid_search(client, embeddings, "What are the degree programmes offered?", top_k=5)
for idx, r in enumerate(results):
  print(f"\n🏆 #{idx + 1} [Page {r['page']}] (Fused Score: {r['score']:.4f})")
  print(f"📌 Matched snippet: {r['matched_snippet'][:150]}...")
  print(f"📄 Parent context (first 200 chars): {r['parent_context'][:200]}...")


--- Test query ---

🏆 #1 [Page 12] (Fused Score: 0.6250)
📌 Matched snippet: Key to understand the Structure of Degree Programmes offered by FCT

Introduction to Organization of the Degree Programmes

Academic programmes of the...
📄 Parent context (first 200 chars): Key to understand the Structure of Degree Programmes offered by FCT

Introduction to Organization of the Degree Programmes

Academic programmes of the Faculty of Computing and Technology are organized...

🏆 #2 [Page 14] (Fused Score: 0.5000)
📌 Matched snippet: Course Structure for the BICT Honours Degree Programme

Credit Distribution of the Course Structure - BICT Honours Degree Programme.

Level Credits fo...
📄 Parent context (first 200 chars): Course Structure for the BICT Honours Degree Programme

Credit Distribution of the Course Structure - BICT Honours Degree Programme.

Level Credits for Compulsory Courses Credits for Optional Courses ...

🏆 #3 [Page 24] (Fused Score: 0.4444)
📌 Matched snippet: Course Structure for 

In [ ]:
results = hybrid_search(client, embeddings, "CSCI 32042", top_k=5)
for idx, r in enumerate(results):
  print(f"\n🏆 #{idx + 1} [Page {r['page']}] (Fused Score: {r['score']:.4f})")
  print(f"📌 Matched snippet: {r['matched_snippet'][:150]}...")
  print(f"📄 Parent context (first 200 chars): {r['parent_context'][:200]}...")


🏆 #1 [Page 36] (Fused Score: 0.5000)
📌 Matched snippet: CSCI 32012 Theory of Automation csc 12013 2 GF EG 1G eG
CSCI 32022 Human Computer Interaction CSCI 21042 2 GS & € ec <€
CSCI 32032 Research Methodolog...
📄 Parent context (first 200 chars): Course Structure for the B.Sc. Honours in Computer Science Degree Programme

CSCI 32012 Theory of Automation csc 12013 2 GF EG 1G eG
CSCI 32022 Human Computer Interaction CSCI 21042 2 GS & € ec <€
CSC...

🏆 #2 [Page 35] (Fused Score: 0.5000)
📌 Matched snippet: CSCI 31082 Systems and Network Administration CSCI 21023,...
📄 Parent context (first 200 chars): Course Structure for the B.Sc. Honours in Computer Science Degree Programme

CSCI 31014 Mathematics for Computer Science Ill csct 12013 *s © & £€ Ce €
CSCI 31022 Machine Learning and Pattern Recogniti...

🏆 #3 [Page 33] (Fused Score: 0.4444)
📌 Matched snippet: ~ CSCI 22012 Statistics for Decision Making Cc 2 CSCI 21013
"7 CSCI 22022 Advanced Operating Systems S 2 CSCI 12052
i CSCI 22032 Objec